> **Status: reference implementation, not the wired-in step 5.**
> The pipeline's step 5 is `step5_JoinNTStopsWithOSM.ipynb`, which runs against
> step 4's ONTD→OSM output so ONTD's country and ids carry through to the seed.
> This notebook matches the schedule directly against the classified OSM
> stations (step 3b) instead. Its geography-first strategy, medoid coordinate
> resolution and confidence labels were folded into the wired-in notebook; it is
> kept because it is the cleaner standalone Stage-B matcher and useful for
> cross-checking. Outputs go to `data/step5_output_*.csv` (see README inventory).


# Step 5 — match current night train stops to OSM stations

See `README.md`'s "Design background" Stage B section and its pipeline overview. Output of this notebook: one confirmed OSM match per current night train stop (from the night train database `stop_times` export), plus review reports for anything the pipeline can't decide on its own (never a silent drop, per the design doc's keep-it-in / auditable principles).

**Note for step 4 (ONTD merge):** the coordinates-first / names-second matcher built here is written to be reused — step 4 matches `data/bahnhoefe_stops_sorted.csv` (ONTD) against the same classified OSM stations with the same approach, per the design doc's "write the matcher once" rule. Use this notebook as the template.

**Prerequisite:** `data/step3b_output_osm_stations_classified.csv` from step 3b.


In [1]:
import re

import geopandas as gpd
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from unidecode import unidecode

from data_sources import ensure_local

## 1. Load & clean

The old OSM export used to carry one stray row where every column literally equalled its own header (a header line that had leaked into the data from a concatenated export). `step3b_classify_stations.ipynb` writes a clean file, so this shouldn't recur — the guard below is kept as a cheap defensive check rather than removed.

In [2]:
# step3b writes plain "." decimals via pandas' default to_csv — no decimal="," here.
# The legacy schedule export below genuinely uses German-locale "," decimals, so it keeps the argument.
osm = pd.read_csv(
    ensure_local("step3b_output_osm_stations_classified.csv"), low_memory=False
)
osm = osm[osm["stop_id"] != "stop_id"]  # drop embedded duplicate header row(s), if any

schedule = pd.read_csv(ensure_local("B-o-T_DataBase_stop_times.csv"), decimal=",")
schedule = schedule[
    ["train_stop_id", "trip_id", "stop_sequence", "stop_id", "stop_lat", "stop_lon"]
]

osm["stop_lat"] = pd.to_numeric(osm["stop_lat"], errors="coerce")
osm["stop_lon"] = pd.to_numeric(osm["stop_lon"], errors="coerce")
schedule["stop_lat"] = pd.to_numeric(schedule["stop_lat"], errors="coerce")
schedule["stop_lon"] = pd.to_numeric(schedule["stop_lon"], errors="coerce")

In [3]:
# A handful of schedule rows have no coordinates at all (arrival/departure-only marker rows).
# They can't be geo-matched, so drop them here rather than let them silently fail the join later.
rows_before = len(schedule)
schedule = schedule.dropna(subset=["stop_lat", "stop_lon"])
print(f"Dropped {rows_before - len(schedule)} schedule rows with missing coordinates")

Dropped 197 schedule rows with missing coordinates


## 2. Collapse the schedule to one row per stop

`schedule` has one row per (trip, stop) — the same physical stop appears once per train that calls there, and duplicate reports of the same stop don't always agree on coordinates (copy/paste errors in the source data, e.g. one `Bern` entry carrying Amsterdam's coordinates).

We resolve this with a **medoid**: for each `stop_id`, pick the reported coordinate that is closest (in total distance) to all other reports of that same stop. That's robust to a single bad outlier report, unlike a plain mean/first-row pick. Spread between reports is also recorded, so stops where reports disagree by more than a sane GPS-jitter margin are flagged for manual review instead of silently resolved — a two-report split (like the Bern/Amsterdam case) is exactly the situation the medoid can't resolve reliably from the numbers alone.

In [4]:
COORD_CONFLICT_THRESHOLD_M = (
    100  # above this spread, treat as a genuine data conflict, not GPS jitter
)


def resolve_stop_coords(group: pd.DataFrame) -> pd.Series:
    """Pick one representative coordinate per stop_id via medoid, and flag disagreement."""
    lat = group["stop_lat"].to_numpy()
    lon = group["stop_lon"].to_numpy()

    if len(lat) == 1:
        return pd.Series(
            {
                "stop_lat": lat[0],
                "stop_lon": lon[0],
                "n_reports": 1,
                "coord_spread_m": 0.0,
            }
        )

    # Equirectangular approximation is accurate enough at station-cluster scale (a few km at most)
    lat_m = lat * 111_320
    lon_m = lon * 111_320 * np.cos(np.radians(lat.mean()))
    pairwise_dist = np.sqrt(
        (lat_m[:, None] - lat_m[None, :]) ** 2 + (lon_m[:, None] - lon_m[None, :]) ** 2
    )

    medoid_idx = pairwise_dist.sum(axis=1).argmin()
    return pd.Series(
        {
            "stop_lat": lat[medoid_idx],
            "stop_lon": lon[medoid_idx],
            "n_reports": len(lat),
            "coord_spread_m": pairwise_dist.max(),
        }
    )


stops = schedule.groupby("stop_id").apply(resolve_stop_coords).reset_index()
print(f"Schedule collapsed from {len(schedule)} rows to {len(stops)} unique stops")

Schedule collapsed from 3672 rows to 601 unique stops


In [5]:
coord_conflicts = stops[
    stops["coord_spread_m"] > COORD_CONFLICT_THRESHOLD_M
].sort_values("coord_spread_m", ascending=False)
coord_conflicts.to_csv(
    "data/step5_output_schedule_coord_conflicts_report.csv", index=False
)
print(
    f"{len(coord_conflicts)} stops have disagreeing coordinate reports — see step5_output_schedule_coord_conflicts_report.csv"
)
coord_conflicts

10 stops have disagreeing coordinate reports — see schedule_coord_conflicts_report.csv


,stop_id,stop_lat,stop_lon,n_reports,coord_spread_m
541,Valladolid-Campo Grande,41.64305,-4.726651,3.0,775972.406722
282,Lausanne,46.51708,6.629193,6.0,665402.809247
170,Fribourg / Freiburg,46.80321,7.151174,4.0,642798.330234
50,Bern,46.94839,7.436390,6.0,633545.823521
585,Zürich HB,47.37818,8.540212,24.0,620178.372387
379,Olten,47.34996,7.903703,4.0,601922.947366
26,Baden,47.47656,8.307640,2.0,596039.770242
12,Antwerpen Centraal,51.21728,4.421142,12.0,133573.688673
440,Rotterdam Centraal,51.92438,4.469746,12.0,58615.180006
457,Schipol Airport,52.30959,4.762805,9.0,12148.874415


## 3. Normalize names before comparing them

`Name (ASCII)` on the OSM side is already transliterated, but the schedule's `stop_id` (which holds the station *name*, not an id) is not — e.g. `București Nord`, `Timişoara Nord`. Comparing an accented name against a transliterated one drags the fuzzy match score down for no real reason. Normalize both sides the same way — strip accents, lowercase, collapse punctuation — before scoring.

In [6]:
def normalize_name(name: str) -> str:
    ascii_name = unidecode(str(name)).lower()
    return re.sub(r"[^a-z0-9]+", " ", ascii_name).strip()


osm["name_norm"] = osm["stop_name"].map(normalize_name)
stops["name_norm"] = stops["stop_id"].map(normalize_name)

## 4. Geo-match: OSM stations vs. schedule stops

Two fixes here vs. the previous approach:

- **Projection.** `EPSG:25832` is a UTM zone centered on Germany — fine for DE, but it distorts   badly for anything far from that zone (this network spans from Portugal to Ukraine).   `EPSG:3035` (ETRS89-LAEA Europe) is the standard pan-European equal-area CRS and keeps meter-based   distances accurate continent-wide, so it's the right choice for a Europe-scale nearest-neighbour join.
- **Nearest-neighbour join.** `sjoin_nearest` with a `max_distance` replaces the buffer+`contains`   join — it directly returns the true point-to-point distance (no need to re-look-up geometries   afterwards) and naturally gives one nearest candidate per schedule stop instead of every OSM   station within a fixed radius.

In [7]:
MAX_MATCH_RADIUS_M = 1_500  # widen-search ceiling from the design doc's Stage B section (README.md) (start 500 m, widen to 1.5 km)

osm_gdf = gpd.GeoDataFrame(
    osm, geometry=gpd.points_from_xy(osm.stop_lon, osm.stop_lat), crs="EPSG:4326"
).to_crs("EPSG:3035")

stops_gdf = gpd.GeoDataFrame(
    stops, geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat), crs="EPSG:4326"
).to_crs("EPSG:3035")

candidates = gpd.sjoin_nearest(
    stops_gdf,
    osm_gdf,
    max_distance=MAX_MATCH_RADIUS_M,
    distance_col="distance_m",
    how="left",
)
print(
    f"{candidates['index_right'].isna().sum()} schedule stops have no OSM candidate within {MAX_MATCH_RADIUS_M} m"
)

140 schedule stops have no OSM candidate within 1500 m


In [8]:
candidates["name_score"] = candidates.apply(
    lambda r: (
        fuzz.token_sort_ratio(r["name_norm_left"], r["name_norm_right"])
        if pd.notna(r["index_right"])
        else np.nan
    ),
    axis=1,
)

# Both terms are scaled 0-100 so the weights are directly comparable regardless of MAX_MATCH_RADIUS_M
candidates["score"] = 0.7 * candidates["name_score"] + 0.3 * (
    100 - candidates["distance_m"] / (MAX_MATCH_RADIUS_M / 100)
)

## 5. Resolve to one match per schedule stop

`stops` is already unique per `stop_id`, so `sjoin_nearest` already returns at most one OSM candidate per schedule stop (ties broken arbitrarily by geopandas) — no separate dedup step against `index_right` is needed here, and it would only reintroduce the earlier bug of two conflicting `drop_duplicates` passes silently overwriting each other's filtering.

What *can* still happen is the reverse: two differently-named schedule stops both resolve to the **same** OSM station (e.g. `Wien Hbf` and `Wien Hbf (Autoreisezuganlage)`, or genuine duplicate spellings like `Göteborg C` / `Göteborg Centralen station`). That's a real signal worth a look, not something to hide by silently keeping only the higher-scoring one.

In [9]:
matched = candidates[candidates["index_right"].notna()].copy()
unmatched = candidates[candidates["index_right"].isna()].copy()

unmatched[
    ["stop_id_left", "stop_lat_left", "stop_lon_left", "n_reports", "coord_spread_m"]
].to_csv("data/step5_output_unmatched_report.csv", index=False)
print(
    f"{len(unmatched)} schedule stops written to step5_output_unmatched_report.csv (no OSM match within {MAX_MATCH_RADIUS_M} m)"
)

140 schedule stops written to unmatched_report.csv (no OSM match within 1500 m)


In [10]:
same_osm_station = matched[matched.duplicated("stop_id_right", keep=False)].sort_values(
    "stop_id_right"
)
same_osm_station.to_csv(
    "data/step5_output_duplicate_osm_matches_report.csv", index=False
)
print(
    f"{same_osm_station['stop_id_right'].nunique()} OSM stations were matched by more than one schedule stop name"
)
print(
    "-> see step5_output_duplicate_osm_matches_report.csv (likely alternate spellings of the same stop, worth consolidating upstream)"
)

361 OSM stations were matched by more than one schedule stop name
-> see duplicate_osm_matches_report.csv (likely alternate spellings of the same stop, worth consolidating upstream)


## 6. Confidence labelling

Per Stage B: record a confidence label per match instead of a single hard cutoff, so low-confidence matches stay in the output (keep-it-in principle) but are clearly marked for the QGIS review pass.

In [11]:
def confidence_label(row: pd.Series) -> str:
    if row["distance_m"] <= 500 and row["name_score"] >= 85:
        return "exact"
    if row["distance_m"] <= 500 and row["name_score"] >= 60:
        return "geo_name"
    if row["distance_m"] <= 500:
        return "geo_only"
    return "ambiguous"


matched["match_confidence"] = matched.apply(confidence_label, axis=1)
matched["match_confidence"].value_counts()

match_confidence
exact        477
geo_name     213
ambiguous     98
geo_only      97
Name: count, dtype: int64

## 7. Final output

In [12]:
result = matched[
    [
        "stop_id_left",
        "stop_lat_left",
        "stop_lon_left",
        "stop_id_right",
        "stop_name",
        "country",
        "station_mode",
        "mode_rule",
        "distance_m",
        "name_score",
        "score",
        "match_confidence",
        "n_reports",
        "coord_spread_m",
    ]
].rename(
    columns={
        "stop_id_left": "stop_id",
        "stop_lat_left": "stop_lat",
        "stop_lon_left": "stop_lon",
        "stop_id_right": "osm_id",
        "stop_name": "osm_name",
        "country": "osm_country",
        "station_mode": "osm_station_mode",
        "mode_rule": "osm_mode_rule",
    }
)
result.to_csv("data/step5_output_matched_stops.csv", index=False)

In [13]:
print("OSM rows:", len(osm))
print("Schedule rows (raw):", rows_before)
print("Schedule stops (unique):", len(stops))
print("Matched:", len(result))
print("Unmatched:", len(unmatched))
print("Coordinate conflicts flagged:", len(coord_conflicts))
print(
    "OSM stations claimed by >1 schedule stop:",
    same_osm_station["stop_id_right"].nunique(),
)
result["match_confidence"].value_counts()

OSM rows: 92207
Schedule rows (raw): 3869
Schedule stops (unique): 601
Matched: 885
Unmatched: 140
Coordinate conflicts flagged: 10
OSM stations claimed by >1 schedule stop: 361


match_confidence
exact        477
geo_name     213
ambiguous     98
geo_only      97
Name: count, dtype: int64